# M606 Machine Learning – Individual Project

**Name:** Nirjar Dipakbhai Baladha  
**Student ID:** GH1051918  
**Module:** M606 Machine Learning  
**Dataset:** Telco Customer Churn (IBM Sample Dataset)  
**Dataset URL:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn 

**Github repo:** https://github.com/Nirjxr/Customer_Churn-Machine-Learning

Problem Statement

Business Context

One of the most expensive issues of telecommunications companies is customer churn the rate of the subscribers to their service terminating. The industry studies point out that it is five to twenty-five times more expensive to acquire a new customer than an existing customer (Reichheld and Schefter, 2000). To a huge telecom company that has millions of subscribers, even a small increase in the accuracy of churn prediction can mean millions of euros of revenue saved every year.

Why This Problem Matters

The reactive system - acting after a customer has gone is in itself expensive and ineffective. An active, data-driven initiative focused on tracking at-risk customers prior to a churn will enable the business to make an intervention by offering customers specific retention incentives (e.g., contract upgrades, loyalty discounts, priority support). This transforms the company to an offensive retention plan to a defensive one.

Data Collection

The publicly available IBM Telco Customer Churn dataset is used in this project. Such data would be gathered in a real-life scenario out of: CRM systems (customer demographics and contract information), billing systems (monthly payments, payment options), and service usage logs (internet, phone, streaming services).

Formulation of Machine Learning.

This issue is posed as binary classification problem:
Target variable: Churn (Yes = customer left within the last month, No = customer stayed)
Attributes of input: 19 demographic, service-usage, and billing attributes.
Goal: Train a classifier that is used to predict whether any individual customer will churn, so that the retention team can focus on who they should reach out to.

The model should not be considered in isolation, since the and the imbalance between classes (non-churners and churners) and the asymmetry between the cost of false negative and false positive should be taken into account in general and not only with regard to the accuracy.

In [3]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
print("All libraries imported successfully.")

ModuleNotFoundError: No module named 'numpy'

## 2. Data Exploration and Characteristics

### 2.1 Loading and Inspecting the Dataset

In [ ]:
# Load the dataset
df_raw = pd.read_csv('Telco_Customer_Churn.csv')
print(f"Dataset shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Data types and basic info
df_raw.info()

In [ ]:
# Basic descriptive statistics for numerical columns
df_raw.describe()

2.2 Data Quality Issues

On inspection, a number of data quality problems were observed:

customerID: Unique identifier which is not predictive - need to be dropped.
TotalCharges: This Charge is numeric, but stored as a string/object. Quiet spaces lead to failures of pd.to_numeric conversions with NaN values of new customers whose tenure is zero.
SeniorCitizen: Forced as a 0/1 integer- no transformation.
Class imbalance: Churn is not balanced (approximately 26% churners and 74 non-churners), which makes the evaluation of the model and the choice of metrics.

In [ ]:
# Fix TotalCharges: convert to numeric (spaces become NaN)
df = df_raw.copy()
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# These 11 rows are new customers (tenure=0) — drop them
df.dropna(subset=['TotalCharges'], inplace=True)

# Drop customerID
df.drop(columns=['customerID'], inplace=True)

# Encode target
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f"\nCleaned dataset shape: {df.shape}")

In [ ]:
# Class distribution
churn_counts = df['Churn'].value_counts()
print("Class distribution:")
print(churn_counts)
print(f"\nChurn rate: {churn_counts[1] / len(df):.2%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
sns.countplot(x='Churn', data=df, palette='muted', ax=axes[0])
axes[0].set_xticklabels(['No Churn (0)', 'Churn (1)'])
axes[0].set_title('Target Variable Distribution')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(churn_counts, labels=['No Churn', 'Churn'], autopct='%1.1f%%',
            colors=['#4878CF', '#D65F5F'], startangle=90)
axes[1].set_title('Churn Proportion')

plt.tight_layout()
plt.show()

print("""
Discussion: The dataset is moderately imbalanced (~26% churners). 
This means accuracy alone is misleading — a model predicting 'No Churn' 
for every customer would achieve 74% accuracy without learning anything useful.
We will use ROC-AUC and F1-score (weighted) as primary evaluation metrics,
and stratified splits/cross-validation to preserve class proportions.""")

### 2.3 Exploratory Data Analysis (EDA)

In [ ]:
# Numerical feature distributions by churn
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
for ax, col in zip(axes, num_cols):
    sns.histplot(data=df, x=col, hue='Churn', kde=True, bins=30,
                 palette={0: '#4878CF', 1: '#D65F5F'}, ax=ax, alpha=0.6)
    ax.set_title(f'{col} by Churn')
plt.suptitle('Numerical Features vs Churn', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Key categorical features vs Churn
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flatten(), cat_cols):
    churn_rate = df.groupby(col)['Churn'].mean().sort_values(ascending=False)
    churn_rate.plot(kind='bar', ax=ax, color='#D65F5F', edgecolor='white')
    ax.set_title(f'Churn Rate by {col}')
    ax.set_ylabel('Churn Rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.set_ylim(0, 0.6)
plt.suptitle('Churn Rate by Key Categorical Features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numerical features only)
plt.figure(figsize=(6, 4))
corr = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix (Numerical Features)')
plt.tight_layout()
plt.show()

print("""
Key EDA Findings:
• Customers on Month-to-month contracts churn at ~43% vs ~11% for Two-year contracts.
• Fiber optic internet users churn significantly more than DSL users.
• Short-tenure customers (< 12 months) are the highest-risk group.
• Higher MonthlyCharges correlate with increased churn.
• Lack of TechSupport and OnlineSecurity strongly associates with churn.""")

Data Preparation Data Preprocessing and Data Engineering.

Why We’re Doing It This Way
The golden rule of machine learning is easy, do not look at the answers. In cleaning up or transforming our data, we must ensure that all our calculations (such as finding the average, scaling values) are only using the training set. In doing so, we leak data. It is similar to handing a student a practice exam that contains the real questions on the final- the student will do well in it, but hasn't really learned how to solve new problems.

We are employing sklearn.pipeline.Pipeline to keep ourselves honest. This tool is a self-enclosed assembly line, the fit occurs only on the training data and then these same rules are just replicated on the test data.

Our Strategy for Categories
You may be tempted to invoke pd.get_dummies, but this is dangerous as it acts on the whole dataset simultaneously, so it is a prime cause of leakage. Rather, we are just keeping OrdinalEncoder in our pipeline.

We are dealing with the various labels in the following way:

Yes/No Columns: These are mapped simply to 0 or 1.

Multi-choice Categories:Ordinal encoding is being used. Why? Since the tree-based models we are working with do not actually require that the numbers be in order, we only require them to be unique. In our case with Logistic Regression, it copes with these categories perfectly as long as we scale the data appropriately given that the size of our data is this large.

In [ ]:
# ── Define feature sets ──────────────────────────────────────────────────────

# Binary Yes/No columns → map to 0/1
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
internet_binary = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                   'TechSupport', 'StreamingTV', 'StreamingMovies']

# Multi-category columns → OrdinalEncoder
multi_cols = ['gender', 'MultipleLines', 'InternetService', 'Contract', 'PaymentMethod']

# Numerical columns (pass through)
num_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

# Apply binary mapping manually (safe: no fit required for deterministic maps)
map_yes_no = {'Yes': 1, 'No': 0,
              'No phone service': 0, 'No internet service': 0}

df_proc = df.copy()
for col in binary_cols + internet_binary:
    df_proc[col] = df_proc[col].map(map_yes_no)

# Feature engineering: tenure groups (domain knowledge)
df_proc['tenure_group'] = pd.cut(df_proc['tenure'],
                                  bins=[0, 12, 24, 48, 72],
                                  labels=[0, 1, 2, 3]).astype(int)

# Define X and y
FEATURE_COLS = num_cols + binary_cols + internet_binary + multi_cols + ['tenure_group']
X = df_proc[FEATURE_COLS]
y = df_proc['Churn']

print("Feature matrix shape:", X.shape)
print("Target distribution:")
print(y.value_counts(normalize=True).round(3))

In [ ]:
# ── Train / Test Split (stratified, 80/20) ───────────────────────────────────
# IMPORTANT: split BEFORE any encoder fitting to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set  : {X_train.shape[0]} samples")
print(f"Test set      : {X_test.shape[0]} samples")
print(f"Train churn % : {y_train.mean():.2%}")
print(f"Test  churn % : {y_test.mean():.2%}")

## 4. Hyperparameter Training of Models.

### Model Selection Strategy

Three candidate algorithms are tested:
1. **Logistic Regression** - a linear interpretable and computationally efficient baseline.
2. **Random Forest** - is a collection of decision trees that are effective in non-linear relationships and feature interactions.
3. **Gradient Boosting** - A sequential boosting ensemble which has been shown to perform well on tabular classification problems.

The models are implemented in a `Pipeline` with the addition of the OrdinalEncoder (to the multi-category columns) and StandardScaler. This guarantees that there is no data leakage: only the training fold is fitted to the encoder and scaler.

The hyperparameters are optimised using the `GridSearchCV` with 5-fold stratified cross-validation which optimises the ROC-AUC - a metric that is resistant to class imbalance and measures the capability of a model to rank churners higher than non-churners at all thresholds.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Columns needing OrdinalEncoder (multi-category strings)
ord_cols = multi_cols  # ['gender','MultipleLines','InternetService','Contract','PaymentMethod']
# All remaining features are already numeric
pass_cols = num_cols + binary_cols + internet_binary + ['tenure_group']

# Preprocessing sub-pipeline
preprocessor = ColumnTransformer(transformers=[
    ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ord_cols),
    ('num', StandardScaler(), pass_cols)
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = 'roc_auc'

print("Preprocessing pipeline defined.")
print(f"Ordinal-encoded columns : {ord_cols}")
print(f"Scaled numeric columns  : {pass_cols}")

In [ ]:
# ── 4a. Logistic Regression ──────────────────────────────────────────────────
pipe_lr = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'))
])

param_grid_lr = {
    'clf__C': [0.01, 0.1, 1, 10],
    'clf__solver': ['lbfgs', 'liblinear']
}

gs_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=cv, scoring=scoring, n_jobs=-1, verbose=0)
gs_lr.fit(X_train, y_train)

print("Logistic Regression – Best params:", gs_lr.best_params_)
print(f"Logistic Regression – Best CV ROC-AUC: {gs_lr.best_score_:.4f}")

In [ ]:
# ── 4b. Random Forest ────────────────────────────────────────────────────────
pipe_rf = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'))
])

param_grid_rf = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5]
}

gs_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=cv, scoring=scoring, n_jobs=-1, verbose=0)
gs_rf.fit(X_train, y_train)

print("Random Forest – Best params:", gs_rf.best_params_)
print(f"Random Forest – Best CV ROC-AUC: {gs_rf.best_score_:.4f}")

In [ ]:
# ── 4c. Gradient Boosting ────────────────────────────────────────────────────
pipe_gb = Pipeline([
    ('pre', preprocessor),
    ('clf', GradientBoostingClassifier(random_state=RANDOM_STATE))
])

param_grid_gb = {
    'clf__n_estimators': [100, 200],
    'clf__learning_rate': [0.05, 0.1],
    'clf__max_depth': [3, 5]
}

gs_gb = GridSearchCV(pipe_gb, param_grid_gb, cv=cv, scoring=scoring, n_jobs=-1, verbose=0)
gs_gb.fit(X_train, y_train)

print("Gradient Boosting – Best params:", gs_gb.best_params_)
print(f"Gradient Boosting – Best CV ROC-AUC: {gs_gb.best_score_:.4f}")

In [ ]:
# ── Model Comparison ─────────────────────────────────────────────────────────
results = {
    'Logistic Regression': gs_lr.best_score_,
    'Random Forest':       gs_rf.best_score_,
    'Gradient Boosting':   gs_gb.best_score_,
}

results_df = pd.DataFrame.from_dict(results, orient='index', columns=['CV ROC-AUC'])
results_df = results_df.sort_values('CV ROC-AUC', ascending=False)
print("Cross-Validation Results (5-Fold Stratified, ROC-AUC):")
print(results_df.round(4))

plt.figure(figsize=(8, 4))
ax = results_df['CV ROC-AUC'].plot(kind='bar', color=['#2ecc71','#3498db','#e74c3c'][:len(results_df)],
                                    edgecolor='white', width=0.5)
ax.set_ylim(0.75, 0.90)
ax.set_ylabel('ROC-AUC Score')
ax.set_title('Model Comparison — 5-Fold Cross-Validation ROC-AUC')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.4f}', (p.get_x() + p.get_width()/2, p.get_height() + 0.001),
                ha='center', fontsize=10)
plt.tight_layout()
plt.show()

best_name = results_df.index[0]
print(f"\n→ Best model selected: {best_name}")

## 5. Final Model Evaluation on the Held-Out Test Set

The best model (selected purely on cross-validation performance) is evaluated **once** on the test set that was never seen during training or model selection. This single evaluation gives an unbiased estimate of real-world generalisation performance.

In [ ]:
# Select best grid-search object
best_gs = {'Logistic Regression': gs_lr,
           'Random Forest': gs_rf,
           'Gradient Boosting': gs_gb}[best_name]

best_model = best_gs.best_estimator_

# ── Test-set predictions ──────────────────────────────────────────────────────
y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

test_auc  = roc_auc_score(y_test, y_proba)
test_acc  = accuracy_score(y_test, y_pred)
test_f1   = f1_score(y_test, y_pred, average='weighted')

print(f"{'='*50}")
print(f" Final Model : {best_name}")
print(f"{'='*50}")
print(f" ROC-AUC     : {test_auc:.4f}")
print(f" Accuracy    : {test_acc:.4f}")
print(f" F1 (weighted): {test_f1:.4f}")
print(f"{'='*50}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'{best_name}\nConfusion Matrix (Test Set)')

# ROC Curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC-AUC = {test_auc:.3f}')
axes[1].plot([0,1],[0,1],'--', color='gray', label='Random classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'{best_name} — ROC Curve (Test Set)')
axes[1].legend()
plt.tight_layout()
plt.show()

### 5.1 Feature Importance and Model Explainability

In [ ]:
# Feature importance — works for Random Forest and Gradient Boosting
import inspect

clf_step = best_model.named_steps['clf']

if hasattr(clf_step, 'feature_importances_'):
    # Get feature names from the preprocessor
    ord_feat_names = ord_cols
    num_feat_names = pass_cols
    all_feat_names = ord_feat_names + num_feat_names

    importances = clf_step.feature_importances_
    imp_df = pd.DataFrame({'Feature': all_feat_names, 'Importance': importances})
    imp_df = imp_df.sort_values('Importance', ascending=True).tail(12)

    plt.figure(figsize=(9, 6))
    plt.barh(imp_df['Feature'], imp_df['Importance'], color='steelblue', edgecolor='white')
    plt.xlabel('Feature Importance (Mean Decrease in Impurity)')
    plt.title(f'{best_name} — Top Feature Importances')
    plt.tight_layout()
    plt.show()

elif hasattr(clf_step, 'coef_'):
    # Logistic Regression coefficients
    ord_feat_names = ord_cols
    num_feat_names = pass_cols
    all_feat_names = ord_feat_names + num_feat_names

    coef = clf_step.coef_[0]
    coef_df = pd.DataFrame({'Feature': all_feat_names, 'Coefficient': coef})
    coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=True).index).tail(12)

    colors = ['#D65F5F' if c > 0 else '#4878CF' for c in coef_df['Coefficient']]
    plt.figure(figsize=(9, 6))
    plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Logistic Regression Coefficient (log-odds)')
    plt.title('Feature Coefficients — Logistic Regression')
    plt.tight_layout()
    plt.show()

print("""
Interpretation: Features with the highest importance / largest absolute coefficients
are the primary drivers of churn prediction. Positive coefficients (red bars) increase
the probability of churn; negative ones (blue bars) decrease it.""")

### 6. Final Discussion

6.1 Strengths of the Pipeline

Leak-free architecture: The Pipeline + ColumnTransformer architecture guarantees that the OrdinalEncoder and StandardScaler are only fit on the training data of each cross-validation fold. This avoids leaking of data which is a very important correctness requirement.
Model selection based on principle: Three candidate models were optimized using GridSearchCV and 5-fold stratified cross-validation. The highest-performing model was based solely on CV ROC-AUC, and evaluated twice on the held-out set only - not being overfitted to the test set.
Proper evaluation measures: ROC-AUC and weighted F1-score are applied, rather than raw accuracy that is misinformed with imbalanced data (74% No-Churn majority baseline).
Stratification: Each train/test split and cross-validation fold is stratified to maintain the original proportions of classes.

6.2 Limitations

Snapshot data: The data is a one month snapshot. Temporal dynamics (e.g., the volume of complaints that a customer has is on the rise) are not captured.
None of the sentiment or interaction data: There is no such information as call centre transcripts, NPS scores, or social media signals, which can be the most powerful predictors of churn, in this dataset.
Lack of full control of class imbalance: Although the class weight was set to balanced, more violent methods like SMOTE (Chawla et al., 2002) would be even better in enhancing the recall of the minority churn class.

6.3 Business Implications and Recommendations.

According to the importances of the features in the model, the following data-driven recommendations can be made:

Retention campaigns should focus on customers who have had a month to month contract. The churn rates of these customers are the highest (~43%). Churn would be directly impacted by incentives specifically to convert them to annual or two-year contracts.
Target the initial 12 months in office. The most vulnerable are short-term customers. Early churn could be greatly mitigated by having proactive onboarding programmes and early check-in calls.
Check Fiber optic price and quality of service. The churn rate of fiber optic customers is disproportionately high, perhaps because of price sensitivity or service reliability problems.
OnlineSecurity and Bundle Tech Support. Customers who do not have these add-ons churn much more. They could be offered as default inclusions or subsidised add-ons to enhance retention.

6.4 Model Explainability and Deployment.

The model that will be chosen is the Logistic Regression model, as it is fully explainable: every coefficient will reflect the change in the log-odds of churn due to a one-unit change in the feature. Random Forest and Gradient Boosting (tree-based models) provide importances of specific features but are less interpretable at a single prediction level; SHAP values may be utilized to explain the reasons specific to a customer in a production environment.

Deployment readiness: The trained Pipeline object can be serialised with joblib and deployed as a REST API endpoint (e.g. via FastAPI or Flask). The model would re-score the customer base once every month, which would supply a rank of the at-risk list to the CRM dashboard of the retention team. It is highly advisable to conduct a holdout check on more recent data and an A/B test of the retention intervention before rolling out the production.

## References

Chawla, N.V., Bowyer, K.W., Hall, L.O. and Kegelmeyer, W.P. (2002) 'SMOTE: Synthetic Minority Over-sampling Technique', *Journal of Artificial Intelligence Research*, 16, pp. 321–357.

IBM (2018) *Telco Customer Churn Dataset*. Available at: https://www.kaggle.com/datasets/blastchar/telco-customer-churn (Accessed: 18 December 2025).

James, G., Witten, D., Hastie, T. and Tibshirani, R. (2013) *An Introduction to Statistical Learning*. New York: Springer.

Pedregosa, F. et al. (2011) 'Scikit-learn: Machine Learning in Python', *Journal of Machine Learning Research*, 12, pp. 2825–2830.

Reichheld, F.F. and Schefter, P. (2000) 'E-Loyalty: Your Secret Weapon on the Web', *Harvard Business Review*, 78(4), pp. 105–113.